# 🛡️ Détection d'Intrusions Réseau par Apprentissage Automatique
## Inspiré de NVIDIA Morpheus

Ce notebook implémente un système de détection d'intrusions (IDS) en utilisant :
- **Random Forest** (sklearn)
- **XGBoost** (xgboost)

Dataset : **NSL-KDD** — 5 classes : Normal, DoS, Probe, R2L, U2R

---
## 1. Imports et Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, roc_curve, auc, roc_auc_score
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import cross_val_score
import xgboost as xgb
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

# Style des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Noms des classes
CLASS_NAMES = {0: 'Normal', 1: 'DoS', 2: 'Probe', 3: 'R2L', 4: 'U2R'}
CLASS_LABELS = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
COLORS = ['#2ecc71', '#e74c3c', '#f39c12', '#9b59b6', '#e67e22']

print("✅ Tous les imports sont chargés avec succès.")

---
## 2. Chargement et Préparation des Données
*(Réutilisation du pipeline de preparedata.ipynb)*

In [ ]:
# Définition des colonnes NSL-KDD (41 features + label + difficulty)
columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty_level'
]

# Chargement des données
train_df = pd.read_csv('KDDTrain+.txt', names=columns)
test_df = pd.read_csv('KDDTest+.txt', names=columns)

# Supprimer la colonne difficulty_level
train_df = train_df.drop('difficulty_level', axis=1)
test_df = test_df.drop('difficulty_level', axis=1)

print(f"📊 Training set : {train_df.shape[0]:,} lignes, {train_df.shape[1]} colonnes")
print(f"📊 Test set     : {test_df.shape[0]:,} lignes, {test_df.shape[1]} colonnes")

In [ ]:
# Mapping des attaques spécifiques vers les 5 catégories
attack_mapping = {
    # Train set attacks
    'neptune': 'DoS', 'smurf': 'DoS', 'back': 'DoS', 'teardrop': 'DoS', 'pod': 'DoS', 'land': 'DoS',
    'satan': 'Probe', 'ipsweep': 'Probe', 'portsweep': 'Probe', 'nmap': 'Probe',
    'warezclient': 'R2L', 'guess_passwd': 'R2L', 'warezmaster': 'R2L', 'imap': 'R2L',
    'ftp_write': 'R2L', 'multihop': 'R2L', 'phf': 'R2L', 'spy': 'R2L',
    'buffer_overflow': 'U2R', 'rootkit': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R',
    'normal': 'Normal',
    # Test set additional attacks (17 types absents du train set)
    'apache2': 'DoS', 'udpstorm': 'DoS', 'processtable': 'DoS', 'mailbomb': 'DoS', 'worm': 'DoS',
    'mscan': 'Probe', 'saint': 'Probe',
    'snmpgetattack': 'R2L', 'snmpguess': 'R2L', 'sendmail': 'R2L', 'named': 'R2L',
    'xlock': 'R2L', 'xsnoop': 'R2L', 'httptunnel': 'R2L',
    'ps': 'U2R', 'xterm': 'U2R', 'sqlattack': 'U2R'
}

train_df['attack_class'] = train_df['label'].map(attack_mapping)
test_df['attack_class'] = test_df['label'].map(attack_mapping)
train_df = train_df.drop('label', axis=1)
test_df = test_df.drop('label', axis=1)

# Encodage numérique des classes
class_mapping = {'Normal': 0, 'DoS': 1, 'Probe': 2, 'R2L': 3, 'U2R': 4}
train_df['attack_class'] = train_df['attack_class'].map(class_mapping)
test_df['attack_class'] = test_df['attack_class'].map(class_mapping)

# Séparation features / target
y_train = train_df['attack_class']
y_test = test_df['attack_class']

X_train_raw = train_df.drop(['attack_class'], axis=1)
X_test_raw = test_df.drop(['attack_class'], axis=1)

# One-Hot Encoding des colonnes catégorielles
categorical_cols = ['protocol_type', 'service', 'flag']
X_train = pd.get_dummies(X_train_raw, columns=categorical_cols)
X_test = pd.get_dummies(X_test_raw, columns=categorical_cols)

# Alignement des colonnes train/test
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Normalisation (StandardScaler)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_train.columns)

# Calcul des poids de classes
class_weights_array = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights_array))

print(f"\n✅ Preprocessing terminé !")
print(f"   Features shape : {X_train_scaled.shape}")
print(f"   Test shape     : {X_test_scaled.shape}")
print(f"\n📋 Poids des classes :")
for idx, w in class_weights_dict.items():
    print(f"   {CLASS_NAMES[idx]:<10} → {w:.4f}")

---
## 3. Entraînement du Modèle Random Forest 🌲

In [ ]:
print("🌲 Entraînement du Random Forest...")
print("="*50)

rf_model = RandomForestClassifier(
    n_estimators=200,           # Nombre d'arbres
    max_depth=25,               # Profondeur maximale
    min_samples_split=5,        # Min échantillons pour split
    min_samples_leaf=2,         # Min échantillons par feuille
    class_weight=class_weights_dict,  # Gestion du déséquilibre
    random_state=42,
    n_jobs=-1,                  # Utiliser tous les CPU
    verbose=0
)

start_time = time.time()
rf_model.fit(X_train_scaled, y_train)
rf_train_time = time.time() - start_time

# Prédictions
start_time = time.time()
rf_y_pred = rf_model.predict(X_test_scaled)
rf_pred_time = time.time() - start_time

# Probabilités pour les courbes ROC
rf_y_proba = rf_model.predict_proba(X_test_scaled)

print(f"✅ Entraînement terminé en {rf_train_time:.2f}s")
print(f"   Prédiction en {rf_pred_time:.4f}s")
print(f"   Accuracy : {accuracy_score(y_test, rf_y_pred):.4f}")
print(f"   F1 Macro : {f1_score(y_test, rf_y_pred, average='macro'):.4f}")
print(f"   F1 Weighted : {f1_score(y_test, rf_y_pred, average='weighted'):.4f}")

In [ ]:
# Classification Report détaillé pour Random Forest
print("\n📊 RANDOM FOREST — Classification Report")
print("="*60)
print(classification_report(y_test, rf_y_pred, target_names=CLASS_LABELS, digits=4))

In [ ]:
# Matrice de confusion — Random Forest
rf_cm = confusion_matrix(y_test, rf_y_pred)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Matrice brute (nombres)
sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_LABELS,
            yticklabels=CLASS_LABELS, ax=axes[0], linewidths=0.5)
axes[0].set_title('Random Forest — Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Vrai label', fontsize=12)
axes[0].set_xlabel('Label prédit', fontsize=12)

# Matrice normalisée (%)
rf_cm_norm = rf_cm.astype('float') / rf_cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(rf_cm_norm, annot=True, fmt='.2%', cmap='Blues', xticklabels=CLASS_LABELS,
            yticklabels=CLASS_LABELS, ax=axes[1], linewidths=0.5, vmin=0, vmax=1)
axes[1].set_title('Random Forest — Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Vrai label', fontsize=12)
axes[1].set_xlabel('Label prédit', fontsize=12)

plt.tight_layout()
plt.savefig('rf_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Sauvegardé : rf_confusion_matrix.png")

---
## 4. Entraînement du Modèle XGBoost 🚀

In [ ]:
print("🚀 Entraînement de XGBoost...")
print("="*50)

# Conversion des poids de classes pour XGBoost (sample_weight)
sample_weights = y_train.map(class_weights_dict).values

xgb_model = xgb.XGBClassifier(
    n_estimators=200,           # Nombre d'arbres
    max_depth=10,               # Profondeur maximale
    learning_rate=0.1,          # Taux d'apprentissage
    subsample=0.8,              # Sous-échantillonnage
    colsample_bytree=0.8,       # Sous-échantillonnage des features
    objective='multi:softprob', # Classification multiclasse
    num_class=5,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

start_time = time.time()
xgb_model.fit(X_train_scaled, y_train, sample_weight=sample_weights)
xgb_train_time = time.time() - start_time

# Prédictions
start_time = time.time()
xgb_y_pred = xgb_model.predict(X_test_scaled)
xgb_pred_time = time.time() - start_time

# Probabilités pour les courbes ROC
xgb_y_proba = xgb_model.predict_proba(X_test_scaled)

print(f"✅ Entraînement terminé en {xgb_train_time:.2f}s")
print(f"   Prédiction en {xgb_pred_time:.4f}s")
print(f"   Accuracy : {accuracy_score(y_test, xgb_y_pred):.4f}")
print(f"   F1 Macro : {f1_score(y_test, xgb_y_pred, average='macro'):.4f}")
print(f"   F1 Weighted : {f1_score(y_test, xgb_y_pred, average='weighted'):.4f}")

In [ ]:
# Classification Report détaillé pour XGBoost
print("\n📊 XGBOOST — Classification Report")
print("="*60)
print(classification_report(y_test, xgb_y_pred, target_names=CLASS_LABELS, digits=4))

In [ ]:
# Matrice de confusion — XGBoost
xgb_cm = confusion_matrix(y_test, xgb_y_pred)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Matrice brute
sns.heatmap(xgb_cm, annot=True, fmt='d', cmap='Oranges', xticklabels=CLASS_LABELS,
            yticklabels=CLASS_LABELS, ax=axes[0], linewidths=0.5)
axes[0].set_title('XGBoost — Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Vrai label', fontsize=12)
axes[0].set_xlabel('Label prédit', fontsize=12)

# Matrice normalisée
xgb_cm_norm = xgb_cm.astype('float') / xgb_cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(xgb_cm_norm, annot=True, fmt='.2%', cmap='Oranges', xticklabels=CLASS_LABELS,
            yticklabels=CLASS_LABELS, ax=axes[1], linewidths=0.5, vmin=0, vmax=1)
axes[1].set_title('XGBoost — Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Vrai label', fontsize=12)
axes[1].set_xlabel('Label prédit', fontsize=12)

plt.tight_layout()
plt.savefig('xgb_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Sauvegardé : xgb_confusion_matrix.png")

---
## 5. Comparaison des Modèles 📊

In [ ]:
# Tableau comparatif
print("\n" + "="*70)
print("       📊 COMPARAISON : RANDOM FOREST vs XGBOOST")
print("="*70)

comparison_data = {
    'Métrique': [
        'Accuracy',
        'F1-Score (Macro)',
        'F1-Score (Weighted)',
        'Temps d\'entraînement (s)',
        'Temps de prédiction (s)'
    ],
    'Random Forest': [
        f"{accuracy_score(y_test, rf_y_pred):.4f}",
        f"{f1_score(y_test, rf_y_pred, average='macro'):.4f}",
        f"{f1_score(y_test, rf_y_pred, average='weighted'):.4f}",
        f"{rf_train_time:.2f}",
        f"{rf_pred_time:.4f}"
    ],
    'XGBoost': [
        f"{accuracy_score(y_test, xgb_y_pred):.4f}",
        f"{f1_score(y_test, xgb_y_pred, average='macro'):.4f}",
        f"{f1_score(y_test, xgb_y_pred, average='weighted'):.4f}",
        f"{xgb_train_time:.2f}",
        f"{xgb_pred_time:.4f}"
    ]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.set_index('Métrique')
print(comparison_df.to_string())
print("="*70)

In [ ]:
# Comparaison par classe (F1-score)
from sklearn.metrics import precision_recall_fscore_support

rf_precision, rf_recall, rf_f1, _ = precision_recall_fscore_support(y_test, rf_y_pred, average=None)
xgb_precision, xgb_recall, xgb_f1, _ = precision_recall_fscore_support(y_test, xgb_y_pred, average=None)

# DataFrame comparatif par classe
per_class_comparison = pd.DataFrame({
    'Classe': CLASS_LABELS,
    'RF Precision': rf_precision,
    'XGB Precision': xgb_precision,
    'RF Recall': rf_recall,
    'XGB Recall': xgb_recall,
    'RF F1': rf_f1,
    'XGB F1': xgb_f1
})

print("\n📊 Comparaison détaillée par classe :")
print(per_class_comparison.to_string(index=False, float_format='%.4f'))

In [ ]:
# Visualisation comparative — F1-Score par classe
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

x = np.arange(len(CLASS_LABELS))
width = 0.35

# F1-Score
bars1 = axes[0].bar(x - width/2, rf_f1, width, label='Random Forest', color='#3498db', alpha=0.85)
bars2 = axes[0].bar(x + width/2, xgb_f1, width, label='XGBoost', color='#e74c3c', alpha=0.85)
axes[0].set_xlabel('Classe d\'attaque', fontsize=12)
axes[0].set_ylabel('F1-Score', fontsize=12)
axes[0].set_title('F1-Score par Classe', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(CLASS_LABELS, rotation=45)
axes[0].legend()
axes[0].set_ylim(0, 1.1)
axes[0].grid(axis='y', alpha=0.3)

# Precision
bars1 = axes[1].bar(x - width/2, rf_precision, width, label='Random Forest', color='#3498db', alpha=0.85)
bars2 = axes[1].bar(x + width/2, xgb_precision, width, label='XGBoost', color='#e74c3c', alpha=0.85)
axes[1].set_xlabel('Classe d\'attaque', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision par Classe', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(CLASS_LABELS, rotation=45)
axes[1].legend()
axes[1].set_ylim(0, 1.1)
axes[1].grid(axis='y', alpha=0.3)

# Recall
bars1 = axes[2].bar(x - width/2, rf_recall, width, label='Random Forest', color='#3498db', alpha=0.85)
bars2 = axes[2].bar(x + width/2, xgb_recall, width, label='XGBoost', color='#e74c3c', alpha=0.85)
axes[2].set_xlabel('Classe d\'attaque', fontsize=12)
axes[2].set_ylabel('Recall', fontsize=12)
axes[2].set_title('Recall par Classe', fontsize=14, fontweight='bold')
axes[2].set_xticks(x)
axes[2].set_xticklabels(CLASS_LABELS, rotation=45)
axes[2].legend()
axes[2].set_ylim(0, 1.1)
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Comparaison Random Forest vs XGBoost', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Sauvegardé : model_comparison.png")

---
## 6. Courbes ROC Multiclasse 📈

In [ ]:
# Binarisation des labels pour ROC multiclasse
y_test_bin = label_binarize(y_test, classes=[0, 1, 2, 3, 4])
n_classes = 5

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ROC pour Random Forest
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], rf_y_proba[:, i])
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=COLORS[i], lw=2,
                 label=f'{CLASS_LABELS[i]} (AUC = {roc_auc:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('Taux de Faux Positifs (FPR)', fontsize=12)
axes[0].set_ylabel('Taux de Vrais Positifs (TPR)', fontsize=12)
axes[0].set_title('Random Forest — Courbes ROC', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(alpha=0.3)

# ROC pour XGBoost
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], xgb_y_proba[:, i])
    roc_auc = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, color=COLORS[i], lw=2,
                 label=f'{CLASS_LABELS[i]} (AUC = {roc_auc:.3f})')

axes[1].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('Taux de Faux Positifs (FPR)', fontsize=12)
axes[1].set_ylabel('Taux de Vrais Positifs (TPR)', fontsize=12)
axes[1].set_title('XGBoost — Courbes ROC', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=10)
axes[1].grid(alpha=0.3)

plt.suptitle('Courbes ROC Multiclasse (One-vs-Rest)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# AUC global
rf_auc = roc_auc_score(y_test_bin, rf_y_proba, average='macro', multi_class='ovr')
xgb_auc = roc_auc_score(y_test_bin, xgb_y_proba, average='macro', multi_class='ovr')
print(f"\n🎯 AUC Macro (One-vs-Rest) :")
print(f"   Random Forest : {rf_auc:.4f}")
print(f"   XGBoost       : {xgb_auc:.4f}")
print(f"\n💾 Sauvegardé : roc_curves.png")

---
## 7. Feature Importance 🔍

In [ ]:
# Top 20 features les plus importantes pour chaque modèle
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Random Forest Feature Importance
rf_importance = pd.Series(rf_model.feature_importances_, index=X_train.columns)
rf_top20 = rf_importance.nlargest(20)

rf_top20.sort_values().plot(kind='barh', ax=axes[0], color='#3498db', alpha=0.85)
axes[0].set_title('Random Forest — Top 20 Features', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Importance', fontsize=12)
axes[0].grid(axis='x', alpha=0.3)

# XGBoost Feature Importance
xgb_importance = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
xgb_top20 = xgb_importance.nlargest(20)

xgb_top20.sort_values().plot(kind='barh', ax=axes[1], color='#e74c3c', alpha=0.85)
axes[1].set_title('XGBoost — Top 20 Features', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Importance', fontsize=12)
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('Features les plus importantes pour la détection d\'intrusions',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Sauvegardé : feature_importance.png")

In [ ]:
# Features communes dans le Top 20
rf_top_features = set(rf_top20.index)
xgb_top_features = set(xgb_top20.index)
common_features = rf_top_features.intersection(xgb_top_features)

print(f"\n🔍 Analyse des Features Importantes")
print(f"="*50)
print(f"\n📌 Features communes aux deux modèles ({len(common_features)}/{20}) :")
for f in sorted(common_features):
    print(f"   • {f} (RF: {rf_importance[f]:.4f} | XGB: {xgb_importance[f]:.4f})")

print(f"\n📌 Features uniques à Random Forest :")
for f in sorted(rf_top_features - common_features):
    print(f"   • {f} ({rf_importance[f]:.4f})")

print(f"\n📌 Features uniques à XGBoost :")
for f in sorted(xgb_top_features - common_features):
    print(f"   • {f} ({xgb_importance[f]:.4f})")

---
## 8. Sauvegarde des Modèles 💾

In [ ]:
# Sauvegarde des modèles et du scaler
joblib.dump(rf_model, 'rf_model.pkl')
joblib.dump(xgb_model, 'xgb_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("💾 Modèles sauvegardés :")
print("   • rf_model.pkl  — Random Forest")
print("   • xgb_model.pkl — XGBoost")
print("   • scaler.pkl    — StandardScaler")

# Déterminer le meilleur modèle
rf_acc = accuracy_score(y_test, rf_y_pred)
xgb_acc = accuracy_score(y_test, xgb_y_pred)
rf_f1_macro = f1_score(y_test, rf_y_pred, average='macro')
xgb_f1_macro = f1_score(y_test, xgb_y_pred, average='macro')

print(f"\n{'='*50}")
if xgb_f1_macro > rf_f1_macro:
    print(f"🏆 Meilleur modèle : XGBoost (F1 Macro: {xgb_f1_macro:.4f})")
    joblib.dump(xgb_model, 'best_model.pkl')
    print("   → Sauvegardé comme best_model.pkl")
else:
    print(f"🏆 Meilleur modèle : Random Forest (F1 Macro: {rf_f1_macro:.4f})")
    joblib.dump(rf_model, 'best_model.pkl')
    print("   → Sauvegardé comme best_model.pkl")

---
## 9. Résumé & Lien avec NVIDIA Morpheus 🔗

### Résultats clés
Ce notebook a implémenté deux algorithmes de classification supervisée pour la **détection d'intrusions réseau** :

| Aspect | Random Forest | XGBoost |
|--------|--------------|----------|
| Type | Ensemble (Bagging) | Ensemble (Boosting) |
| Avantage | Robuste, peu d'overfitting | Performant sur classes rares |
| Inconvénient | Moins bon sur R2L/U2R | Temps d'entraînement plus long |

### Inspiration NVIDIA Morpheus

Notre approche s'inspire du framework **NVIDIA Morpheus** qui utilise le ML pour la cybersécurité :

1. **Pipeline de données** : Comme Morpheus, nous appliquons un pipeline structuré (ingestion → preprocessing → feature engineering → classification)
2. **Classification supervisée** : Morpheus utilise des modèles ML (dont des forêts aléatoires et des gradient boosters) pour classifier le trafic réseau en temps réel
3. **Détection multi-classes** : Notre système détecte 4 types d'attaques (DoS, Probe, R2L, U2R) + trafic normal, similaire à l'approche multi-catégories de Morpheus
4. **Gestion du déséquilibre** : L'utilisation de class weights reproduit la stratégie de Morpheus pour gérer les attaques rares (U2R = 0.04% du dataset)
5. **Feature importance** : L'analyse des features critiques permet de comprendre quels indicateurs réseau sont les plus révélateurs d'une intrusion

### Différences avec Morpheus
- Morpheus fonctionne en **temps réel** sur GPU (RAPIDS/cuML), notre implémentation est en batch sur CPU
- Morpheus intègre du **deep learning** (autoencoders, LSTM) en plus du ML classique
- Morpheus traite des **flux de données en streaming**, pas des fichiers CSV statiques